# .Imports

In [ ]:
import cv2
import numpy as np
import os
import random
from pathlib import Path
import csv
import pandas as pd
import matplotlib.pyplot as plt 

: 

WARNING!!!

estranhamente, se rodar o "!pip install ultralytics" antes de importar o restante das bibliotecas, da erro.

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

# .Definição do video

In [ ]:
# Defina o caminho para o seu arquivo de vídeo
VIDEO_FOLDER = '/kaggle/input/highway-traffic-videos-dataset/video/' 

In [ ]:
def selecionar_video_aleatorio(folder_path, extensions=['.mp4', '.avi', '.mov']):
    """
    Lista os vídeos na pasta e seleciona um aleatoriamente.
    """
    if not os.path.isdir(folder_path):
        print(f"❌ ERRO: O caminho da pasta '{folder_path}' não é válido ou não existe.")
        return None

    # Lista todos os arquivos na pasta
    all_files = os.listdir(folder_path)
    
    # Filtra apenas os arquivos que terminam com as extensões de vídeo desejadas
    video_files = [f for f in all_files if any(f.lower().endswith(ext) for ext in extensions)]
    
    if not video_files:
        print(f"❌ ERRO: Nenhuma mídia válida ({', '.join(extensions)}) encontrada em '{folder_path}'.")
        return None

    # Seleciona um arquivo de vídeo aleatoriamente
    selected_file = random.choice(video_files)
    video_path = os.path.join(folder_path, selected_file)
    
    print(f"✅ Vídeo Selecionado Aleatoriamente: **{selected_file}**")
    return video_path

def verificar_video(video_path):
    """
    Tenta abrir o arquivo de vídeo e exibe suas principais propriedades.
    Salva o primeiro frame em uma imagem PNG para confirmação visual,
    evitando o erro de 'cv2.imshow' em ambientes sem GUI.
    """
    if video_path is None:
        return

    print(f"\n--- Verificando o arquivo: {Path(video_path).name} ---")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("❌ ERRO: Não foi possível abrir o vídeo (verifique o formato/codec).")
        return

    # Captura as propriedades do vídeo
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Dimensões (Largura x Altura): **{frame_width} x {frame_height}** pixels")
    print(f"FPS (Frames por Segundo): **{fps:.2f}**")
    print(f"Total de Quadros: {total_frames}")
    
    # Leitura e salvamento do primeiro quadro
    ret, frame = cap.read()
    if ret:
        print(f"✅ Primeiro quadro lido com sucesso! Formato: {frame.shape}")
        
        # Cria a pasta de saída e define o caminho do arquivo
        output_dir = "frames_de_teste"
        os.makedirs(output_dir, exist_ok=True)
        
        video_name = Path(video_path).stem
        output_path = os.path.join(output_dir, f"{video_name}_primeiro_quadro.png")
        
        # Salva o frame
        cv2.imwrite(output_path, frame)
        print(f"🖼️ Quadro salvo com sucesso em: **{output_path}**")

    else:
        print("❌ ERRO: Não foi possível ler o primeiro quadro do vídeo.")
    
    cap.release()
    print("------------------------------------------------")

# =========================================================================
# --- EXECUÇÃO PRINCIPAL ---
# =========================================================================
selected_video_path = selecionar_video_aleatorio(VIDEO_FOLDER)
verificar_video(selected_video_path)

# .Definição da Mtriz H

In [ ]:
# --- Bibliotecas já importadas: cv2, numpy ---

global H_matrix
global model
global VEHICLE_CLASSES
global pixel_to_real_world
global selected_video_path

if selected_video_path is None:
    print("❌ ERRO: Caminho do vídeo não definido. Execute a Célula 1.")
    raise SystemExit
    
# 1. Definição dos Pontos (Ajuste esses valores com base no seu quadro!)
# Pontos em Pixels (Source Points)
# Estrutura: [[u1, v1], [u2, v2], [u3, v3], [u4, v4]]
pts_src = np.float32([
    [150, 200], # Ponto A: Mais próximo, esquerda
    [250, 200], # Ponto B: Mais próximo, direita
    [260, 130], # Ponto C: Mais distante, direita
    [190, 130]   # Ponto D: Mais distante, esquerda
])

# Pontos no Mundo Real (Destination Points)
# Estrutura: [[xw1, yw1], [xw2, yw2], [xw3, yw3], [xw4, yw4]]
# Assumindo metros: (0,0) é o ponto mais próximo, 3.5m de largura, 10m de profundidade
pts_dst = np.float32([
    [0, 0], 
    [3.5, 0], 
    [3.5, 50], 
    [0, 50]
])

# 2. Cálculo da Matriz de Homografia
# Retorna H, a matriz 3x3 de transformação
H_matrix, status = cv2.findHomography(pts_src, pts_dst)

if H_matrix is None:
    print("❌ ERRO: Não foi possível calcular a Matriz de Homografia.")
    raise SystemExit
else:
    print("✅ Matriz de Homografia (H) calculada com sucesso!")
    
    # Exemplo de Teste: Converte o ponto A (pixel) para o mundo real (deve ser aproximadamente (0, 0))
    test_point_pixel = np.array([[[250, 200]]], dtype='float32') # Ponto A
    test_point_real = cv2.perspectiveTransform(test_point_pixel, H_matrix)

    x_real = test_point_real[0, 0, 0]
    y_real = test_point_real[0, 0, 1]
    
    model = YOLO('yolov8n.pt') 
VEHICLE_CLASSES = [2, 3, 5, 7] 

# 4. Função para Transformar Ponto
def pixel_to_real_world(pixel_coord, H):
    """Converte coordenadas de pixel (u, v) para coordenadas reais (x, y) em metros."""
    point_pixel = np.array([[pixel_coord]], dtype='float32')
    point_real = cv2.perspectiveTransform(point_pixel, H)
    return point_real[0, 0]

print("✅ Inicialização completa: YOLO carregado, Homografia calculada.")

In [ ]:
# --- A) Carregamento da Matriz de Homografia ---
# (Usaremos a matriz calculada na etapa anterior)
# Substitua pelos seus valores reais

# --- B) Inicialização do Modelo YOLO ---
# Carregue o modelo pré-treinado. 'yolov8n.pt' é o modelo nano (rápido e leve).
model = YOLO('yolov8n.pt') 
# Mapeamento de classes (apenas queremos veículos)
# O COCO dataset tem IDs, precisamos filtrar:
VEHICLE_CLASSES = [2, 3, 5, 7]  # Ex: 2=car, 3=motorcycle, 5=bus, 7=truck
# --- C) Função para Transformar Ponto (Relembre a Homografia) ---
def pixel_to_real_world(pixel_coord, H):
    """Converte coordenadas de pixel (u, v) para coordenadas reais (x, y) em metros."""
    point_pixel = np.array([[pixel_coord]], dtype='float32')
    point_real = cv2.perspectiveTransform(point_pixel, H)
    return point_real[0, 0]

# .Captura de video

In [ ]:
# --- D) Configuração da Captura de Vídeo (CORRIGIDO) ---
video_path = selected_video_path 
global csv_filename
csv_filename = f"data_traffic_analysis_{Path(video_path).stem}.csv"

if video_path is None:
    print("❌ ERRO: O caminho do vídeo está vazio. A célula de seleção aleatória falhou?")
    # Evita que o resto da célula falhe
    raise SystemExit 

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"❌ ERRO: Não foi possível abrir o vídeo: {Path(video_path).name}")
    raise SystemExit 

# Captura do FPS e cálculo do Delta T
FPS = cap.get(cv2.CAP_PROP_FPS) 
DT = 1.0 / FPS
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"✅ Vídeo '{Path(video_path).name}' aberto com sucesso. FPS: {FPS:.2f}")

output_filename = f"processed_{Path(video_path).stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec para MP4 (funciona bem em notebooks)
out = cv2.VideoWriter(output_filename, fourcc, FPS, (W, H))

print(f"🎥 O vídeo processado será salvo em: {output_filename}")

# Dicionário para armazenar o histórico de posição de cada ID
# {id: [(x_real, y_real, timestamp), ...]}
vehicle_position_history = {} 
frame_count = 0

# Inicialização da Lista de Dados para Exportação
# Cabeçalhos: ID, Tempo (s), X Real (m), Y Real (m), Velocidade (KPH)
data_export_list = [
    ['track_id', 'timestamp_s', 'x_real_m', 'y_real_m', 'speed_kph']
]
frame_count = 0

# --- E) Loop Principal ---
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # 1. Detecção (YOLO) e Rastreamento
    results = model.track(frame, persist=True, classes=VEHICLE_CLASSES, verbose=False)
    
    # 2. Processamento dos Resultados do Rastreamento
    if results[0].boxes.id is not None:
        
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            
            # --- Ponto-chave: Escolha do Ponto no Chão ---
            center_x = (x1 + x2) // 2
            bottom_y = y2
            pixel_coord = (center_x, bottom_y)

            # 3. Transformação de Coordenadas (Homografia)
            x_real, y_real = pixel_to_real_world(pixel_coord, H_matrix)
            current_time = frame_count * DT

            # 4. Armazenamento e Cálculo de Velocidade
            if track_id not in vehicle_position_history:
                vehicle_position_history[track_id] = []

            vehicle_position_history[track_id].append((x_real, y_real, current_time))
            
            # Garante que temos pelo menos 2 pontos 
            if len(vehicle_position_history[track_id]) >= 2:
                
                prev_x, prev_y, prev_t = vehicle_position_history[track_id][-2]
                curr_x, curr_y, curr_t = vehicle_position_history[track_id][-1]
                
                # Distância euclidiana 2D (em metros)
                distance_meters = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
                time_elapsed = curr_t - prev_t

                if time_elapsed > 0:
                    speed_mps = distance_meters / time_elapsed 
                    speed_kph = speed_mps * 3.6
                    
                    # 5. Visualização (Desenhar Bounding Box e Velocidade)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"ID:{track_id} | {speed_kph:.1f} KPH"
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                    # Registro dos Dados para Exportação
                    # Registra a posição atual e a velocidade calculada para o ID
                    data_export_list.append([
                        track_id,
                        curr_t,      # Tempo em segundos
                        curr_x,      # X Real (metros)
                        curr_y,      # Y Real (metros)
                        speed_kph    # Velocidade em KPH
                    ])

    # Limpar histórico para evitar uso excessivo de memória (opcional)
    
    out.write(frame)
    frame_count += 1

cap.release()
out.release()
print(f"✅ Processamento concluído. Vídeo salvo como {output_filename}")

# Exportação para Arquivo CSV
csv_filename = f"data_traffic_analysis_{Path(video_path).stem}.csv"

with open(csv_filename, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(data_export_list)

print(f"📊 Dados de velocidade exportados para: {csv_filename}")

# .Análise e Vizualização

In [ ]:
# --- CÉLULA NOVA: ANÁLISE ESTATÍSTICA ---
# ATENÇÃO: Carrega o nome do arquivo da variável global
try:
    csv_filename_for_analysis = csv_filename
except NameError:
    print("❌ ERRO: A variável 'csv_filename' não foi definida. Rode a célula de Processamento (Loop Principal) primeiro.")
    raise SystemExit

try:
    # 1. Carregar os dados
    df = pd.read_csv(csv_filename_for_analysis)
    
    print(f"✅ Dados carregados com sucesso do arquivo: {csv_filename_for_analysis}")
    print(f"Total de registros de velocidade: {len(df)}")
    
    # --- 2. Análise Estatística ---
    
    # Filtra dados para remover velocidades muito baixas, que geralmente indicam erros 
    # de rastreamento no início/fim ou carros parados (ajuste este valor, ex: > 5 KPH)
    df_filtered = df[df['speed_kph'] > 5].copy() 
    
    print("\n--- Estatísticas Descritivas (KPH) ---")
    print(df_filtered['speed_kph'].describe())
    
    # 3. Análise por Veículo (ID)
    print("\n--- Velocidade Média por Veículo (ID) ---")
    
    # Agrupa por ID e calcula a média de velocidade
    vehicle_summary = df_filtered.groupby('track_id')['speed_kph'].agg(['mean', 'count'])
    vehicle_summary = vehicle_summary[vehicle_summary['count'] > 5] # Exclui IDs rastreados por poucos quadros
    vehicle_summary = vehicle_summary.sort_values(by='mean', ascending=False)
    
    total_unique_vehicles = df['track_id'].nunique()
    print(f"Total de veículos únicos rastreados (incluindo curtos): {total_unique_vehicles}")
    print(f"Total de veículos com rastreamento longo (>5 quadros): {len(vehicle_summary)}")
    print(vehicle_summary.head(10))
    
    # 4. Visualização dos Dados (Opcional, mas muito útil)
    plt.figure(figsize=(10, 6))
    plt.hist(df_filtered['speed_kph'], bins=20, edgecolor='black')
    plt.title('Distribuição de Frequência das Velocidades')
    plt.xlabel('Velocidade (KPH)')
    plt.ylabel('Frequência')
    plt.grid(axis='y', alpha=0.7)
    plt.show()

except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{csv_filename_for_analysis}' não encontrado. Verifique o nome do arquivo ou se a célula anterior falhou.")